In [1]:
### Import e setup dei moduli
import sys
import json
from pathlib import Path

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.sidecar_manager import SidecarManager
from src.graph_builder import GraphBuilder
from src.chunk_widget import ChunkGraphWidget
from src.distance_reranker import DistanceReranker
from src.config import QDRANT_URL, COLLECTION_NAME, LLM_MODEL, RERANKING_TOP_N, EMBEDDING_MODEL, RETRIEVAL_TOP_K
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from langchain_ollama import ChatOllama
from langchain_community.embeddings import FastEmbedEmbeddings

# Inizializzazione moduli
embeddings = FastEmbedEmbeddings(model_name=EMBEDDING_MODEL)
qdrant_client = QdrantClient(url=QDRANT_URL)
vector_store = QdrantVectorStore(client=qdrant_client, collection_name=COLLECTION_NAME, embedding=embeddings)
sidecar = SidecarManager(project_root / "data/processed/test/sidecar_04_02T.json")
llm = ChatOllama(model=LLM_MODEL, temperature=0.0)

/tmp/ipykernel_104827/2808188408.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import FastEmbedEmbeddings


In [3]:
### Esecuzione Query Baseline (Qt) e Rendering Grafo
queries_path = project_root / "data/queries/ds1/eval_queries.json"
with open(queries_path, "r", encoding="utf-8") as f:
    benchmark_queries = json.load(f)

target_id = ["Q_SINGLE_01"]
#target_id = ["Q_GLOBAL_01"]
matched = [q for q in benchmark_queries if q.get("id") in target_id]
query = matched[0]["query"]

# Recupero candidate chunk da Qdrant
raw_candidates = vector_store.similarity_search_with_score(query, k=RETRIEVAL_TOP_K)

# Costruzione Grafo con distanze da Qdrant
builder = GraphBuilder()
for doc, score in raw_candidates:
    chunk_id = f"{doc.metadata.get('doc_id', 'doc')}_chunk_{doc.metadata.get('chunk_index', 0)}"
    builder.add_chunk_node(chunk_id, doc.page_content)

# Aggiunta archi basati sulla similarity tra i nodi recuperati
for i, (doc1, score1) in enumerate(raw_candidates):
    id1 = f"{doc1.metadata.get('doc_id', 'doc')}_chunk_{doc1.metadata.get('chunk_index', 0)}"
    for doc2, score2 in raw_candidates[i+1:]:
        id2 = f"{doc2.metadata.get('doc_id', 'doc')}_chunk_{doc2.metadata.get('chunk_index', 0)}"
        builder.add_relation(id1, id2, similarity=(score1 + score2) / 2)

# Visualizzazione Widget D3.js
widget = ChunkGraphWidget(sidecar_path="data/processed/test_sidecar.json")
widget.load_graph(builder.to_json_data())
widget

In [ ]:
# Esecuzione Re-ranking e Risposta LLM (Qt+1)
#>> Da eseguire DOPO aver trascinato/allontanato un nodo nel grafico sopra)

reranker = DistanceReranker(sidecar_manager=sidecar)
reranked_chunks = reranker.rerank(raw_candidates, top_n=RERANKING_TOP_N)

# Stampa esito filtri
print("--- CHUNK SOPRAVVISSUTI AL RE-RANKING ---")
context_texts = []
for item in reranked_chunks:
    print(f"ID: {item['chunk_id']} | Final Score: {item['final_score']:.4f} | Penalties: {item['applied_penalties']}")
    context_texts.append(item['raw_doc'].page_content)

# Generazione risposta finale Ollama
context = "\n\n".join(context_texts)
prompt = f"Rispondi alla domanda basandoti solo sul contesto fornito.\n\nContesto:\n{context}\n\nDomanda: {query}"
response = llm.invoke(prompt)

print("\n--- RISPOSTA GENERATA DA OLLAMA ---")
print(response.content)